# 03 — Geometric Biomarkers
## Principal Angle Collapse and Trajectory Tangling as Signatures of WM Failure

---

This is the scientific core of the anchor project. The question:

> **Does the geometry of prefrontal population dynamics signal working memory failure before it happens?**

Two geometric metrics:
1. **Principal angle θ_min(t)**: Are representations of different task states getting confused?
2. **Trajectory tangling Q(t)**: Is the dynamical system becoming noise-sensitive?

Both should be low during successful WM maintenance. Both should be elevated during high-load conditions and target detection events that stress the system.

**Read before this notebook:**
- Russo et al. (2018) — *Motor cortex embeds muscle-like commands in an untangled population response* — Neuron (in your reading list)
- Cunningham & Yu (2014) — *Dimensionality reduction for large-scale neural recordings* — Nat Neurosci

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../../scripts').resolve()))
from ieeg_utils import (
    load_subject, preprocess, high_gamma_power, epoch_data, baseline_normalize,
    reject_bad_channels, pca_project, principal_angles, trajectory_tangling
)

# Load previously computed latent trajectories
# If you saved them in Module 2, load here. Otherwise re-run Module 2 first.
# mu_np shape: (n_trials, n_times, latent_dim)
# Placeholder — replace with actual load:
# mu_np = np.load('../../outputs/mu_trajectories_al.npy')

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

---
## 1. Principal Angles — Measuring Representational Separation

**The setup:** For each time window during the trial, compute the subspace occupied by the population trajectory for condition A vs. condition B. The principal angle between these subspaces tells us how well the brain can distinguish the two states.

**Three comparisons to make:**
1. 0-back vs 2-back subspaces — load effect (should be large θ = good separation)
2. Target vs non-target within 2-back — detection effect
3. Within-condition across time — does the subspace drift during maintenance?

**How to define subspaces:**
For each condition, take the latent trajectories across all trials in a time window.
Stack them: matrix of shape `(n_trials_cond × n_times_window, latent_dim)`.
Take top-k SVD vectors as the subspace basis.

In [ ]:
# ─── Compute time-resolved principal angles ───────────────────────────────────
# For each time bin, compute θ_min between condition pairs

# Assumes mu_np: (n_trials, n_times, latent_dim) from Module 2
# and task_id: (n_trials,)

def time_resolved_angles(mu, task_id, cond_a, cond_b,
                          window_ms=200, step_ms=50, srate=1200, k=3):
    """
    Compute principal angles between cond_a and cond_b subspaces
    in sliding time windows.

    Args:
        mu      : (n_trials, n_times, latent_dim)
        window_ms : sliding window size in ms
        step_ms : step size in ms
        k       : subspace dimensionality

    Returns:
        t_centers : center of each window (in time points)
        angles    : (n_windows, k) principal angles in degrees
    """
    n_trials, n_times, d = mu.shape
    win  = int(window_ms * srate / 1000)
    step = int(step_ms   * srate / 1000)

    mask_a = task_id == cond_a
    mask_b = task_id == cond_b

    t_centers = []
    all_angles = []

    for t_start in range(0, n_times - win, step):
        t_end = t_start + win
        t_centers.append((t_start + t_end) // 2)

        # Subspace A: stack trials × time in window → (n_a * win, d)
        A = mu[mask_a, t_start:t_end, :].reshape(-1, d)
        B = mu[mask_b, t_start:t_end, :].reshape(-1, d)

        # Get top-k directions via SVD (columns of Vt.T = principal directions)
        _, _, VtA = np.linalg.svd(A - A.mean(0), full_matrices=False)
        _, _, VtB = np.linalg.svd(B - B.mean(0), full_matrices=False)

        ang = principal_angles(VtA[:k].T, VtB[:k].T)
        all_angles.append(ang * 180 / np.pi)  # convert to degrees

    return np.array(t_centers), np.array(all_angles)


# You'll run this after loading mu_np from Module 2
# t_centers, angles_0v2 = time_resolved_angles(mu_np, task_id, 0, 2)
# t_centers, angles_1v2 = time_resolved_angles(mu_np, task_id, 1, 2)
print('Function defined. Run after loading mu_np from Module 2.')

---
## 2. Trajectory Tangling — Measuring Dynamical Instability

**Tangling (Russo et al. 2018):**
$$Q(t) = \max_{t'} \frac{\|\dot{z}(t) - \dot{z}(t')\|^2}{\|z(t) - z(t')\|^2 + \varepsilon}$$

High Q(t) at time t means: **similar states at t and t' lead to very different velocities** — the system is sensitive to noise. This is the opposite of what a good WM system wants.

**The hypothesis:**
- During 0-back: Q should be low (no WM load, system is relaxed)
- During 2-back maintenance: Q should be low most of the time (stable attractor)
- At target presentation in 2-back: Q should spike (the system must update, momentarily destabilizing)
- If WM is failing: Q should be elevated throughout maintenance

**Important:** Tangling is computed on the *latent trajectory*, not the raw data. This is why Module 2 must come first — we need a smooth, low-dimensional representation where velocity is meaningful.

In [ ]:
# ─── Compute tangling per trial ───────────────────────────────────────────────

def trial_tangling(mu, task_id, epsilon=1e-3):
    """
    Compute trajectory tangling Q(t) for each trial.

    Strategy: Pool all trajectories together (across all conditions)
    to compute the max over all pairs (t, t') globally.
    This is the Russo et al. approach.

    Returns:
        Q : (n_trials, n_times) tangling at each time point per trial
    """
    n_trials, n_times, d = mu.shape

    # Flatten to (n_trials * n_times, d) for global comparison
    Z    = mu.reshape(-1, d)   # (N, d) where N = n_trials * n_times

    # Compute velocity via finite differences
    Zdot = np.zeros_like(Z)
    # For each trial separately (don't mix across trial boundaries)
    for trial in range(n_trials):
        t0 = trial * n_times
        t1 = t0 + n_times
        z  = Z[t0:t1]          # (n_times, d)
        zd = np.zeros_like(z)
        zd[1:-1] = (z[2:] - z[:-2]) / 2.0  # central difference
        Zdot[t0:t1] = zd

    # Compute Q for each time point (expensive — can subsample t' for speed)
    Q = np.zeros(len(Z))
    for i in range(len(Z)):
        dstate = Z - Z[i]                        # (N, d)
        dvel   = Zdot - Zdot[i]                  # (N, d)
        num    = (dvel  ** 2).sum(axis=1)
        den    = (dstate ** 2).sum(axis=1) + epsilon
        Q[i]   = (num / den).max()

    return Q.reshape(n_trials, n_times)


# NOTE: This is O(N²) in the number of time points — will be slow for large datasets.
# For speed, subsample t' or use a KD-tree. But run it first to understand the metric.
print('Function defined.')
print('Runtime warning: for 150 trials × 2040 time points, this is ~300M comparisons.')
print('Recommend: run on subset first (e.g., 30 trials) to verify behavior.')

In [ ]:
# ─── Plot tangling by condition ────────────────────────────────────────────────
# After computing Q_all = trial_tangling(mu_np, task_id):

# Q_all shape: (n_trials, n_times)
# times shape: (n_times,)

# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
#
# cond_colors = {0: 'royalblue', 1: 'seagreen', 2: 'crimson'}
# for cond in [0, 1, 2]:
#     mask     = task_id == cond
#     mean_Q   = Q_all[mask].mean(axis=0)
#     sem_Q    = Q_all[mask].std(axis=0) / np.sqrt(mask.sum())
#     axes[0].plot(times, mean_Q, color=cond_colors[cond], lw=2, label=TASK_CODES[cond])
#     axes[0].fill_between(times, mean_Q-sem_Q, mean_Q+sem_Q,
#                          color=cond_colors[cond], alpha=0.15)
#
# axes[0].set_xlabel('Time from stimulus onset (s)')
# axes[0].set_ylabel('Trajectory tangling Q(t)')
# axes[0].set_title('Tangling by N-back load')
# axes[0].legend()
#
# # Target vs non-target in 2-back
# for ttype in [1, 2]:
#     mask   = (task_id == 2) & (tgt_id == ttype)
#     mean_Q = Q_all[mask].mean(axis=0)
#     axes[1].plot(times, mean_Q, lw=2, label=TARGET_CODES[ttype])
# axes[1].set_xlabel('Time from stimulus onset (s)')
# axes[1].set_ylabel('Q(t)')
# axes[1].set_title('Tangling: target vs non-target (2-back)')
# axes[1].legend()

print('Plot code ready — uncomment after computing Q_all')

---
## ✏️ Exercises

### A — Before Running: Predict the Shape of Q(t)

Based on what you know about the N-back task:
1. At what time (relative to stimulus onset) do you expect Q(t) to peak? Justify.
2. Should Q(t) be higher for target trials or non-target trials in 2-back? Why?
3. Should Q(t) increase monotonically with N-back level (0 < 1 < 2-back), or is the relationship more complex? Think about what each condition requires computationally.

Write your predictions, then check against the data.

### B — ε Sensitivity
The ε term prevents division by zero when z(t) = z(t'). Try ε = 1e-1, 1e-2, 1e-3, 1e-4. How sensitive is Q(t) to this choice? What's the right value and why?

### C — Latent Dimensionality Effect
Run the analysis with latent_dim = 4, 8, 16 (retrain the VAE). How does Q(t) change? At what dimensionality do the results stabilize? What does this tell you about the effective dimensionality of the WM dynamics?

### D — The Combined Figure
Create a two-panel figure showing:
- Top: θ_min(t) for 0-back vs 2-back
- Bottom: Q(t) for target vs non-target in 2-back

Do they tell the same story or different stories? This is Figure 2 of your paper.

---
## Write: `notes/trajectory_tangling.md`

Derive Q(t) from scratch. Explain why high Q(t) is bad. Connect to the motor cortex result from Russo et al. Explain how your PFC result confirms or disconfirms the motor cortex finding.

## Next: `04_system_id/04_dmd_and_linear_dynamics.ipynb`